In [0]:
import mlflow
from pyspark.sql import functions as F

df_gold = spark.read.table("gold_credit_features")
print(f"總筆數：{df_gold.count()}")

model_uri = "models:/credit_risk_model/1"
predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri)

feature_cols = [c for c in df_gold.columns if c != 'default']

df_predictions = df_gold.withColumn(
    "predicted_default",
    predict_udf(*[F.col(c) for c in feature_cols])
).withColumn(
    "prediction_date",
    F.current_date()
)

df_predictions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_predictions")

display(df_predictions.select("age", "income", "credit_score", "default", "predicted_default", "prediction_date").limit(10))